# 02 · Backtest results

Five models, one protocol, identical target points.

**The protocol.** Rolling origin, expanding window. For each of the last 52 weeks: train on
everything up to 04:00 UTC, forecast the following 24 hours, step back 7 days, refit from
scratch. 52 folds × 24 hours = 1,248 scored points per model.

**Why not a random train/test split.** A random split lets a model train on Tuesday and
Thursday to predict Wednesday. Every autocorrelated series scores brilliantly that way and
the score means nothing. Time-series validation has to cut on time, and the cut has to move.

**Why MASE.** MAPE alone cannot say whether a model is *useful*. MASE divides a model's MAE
by seasonal naive's MAE **on the same target points**, so 1.0 is exactly "no better than
assuming this week looks like last week" and anything above 1.0 is a model that should not
be deployed.

In [ ]:
import sys

sys.path.insert(0, "..")

import pandas as pd
from IPython.display import Image, display

from src import report, viz
from src.backtest import build_summary, load_results, render_summary

viz.apply_theme()
MODEL = "lgbm"

summary = build_summary(list(viz.MODEL_ORDER))
print(render_summary(summary))

## The comparison table

Read the MASE column first. Everything at or above 1.0 lost to a two-line baseline.

In [ ]:
display(Image(report.plot_model_comparison(summary)))

## What the forecast actually looks like

The best and worst folds by MAE, so the aggregate percentage has a shape attached to it.
The worst fold is the more informative of the two — it shows *how* the model fails, not just
that it does.

In [ ]:
results = load_results(MODEL)
display(Image(report.plot_best_worst_folds(results, MODEL)))

## Where the error lives

A single headline MAPE hides the operationally important question: *when* is the model
unreliable? Two cuts — hour × month, and hour across models.

In [ ]:
display(Image(report.plot_error_heatmap(results, MODEL)))
display(Image(report.plot_error_by_hour(list(summary.index))))

### Grouped error tables

The weekday/weekend split is computed from the **dense** backtest (a fold per day for the last
year), not the 52-fold headline run: a 7-day step lands every fold on the same weekday, so the
headline protocol cannot answer that question. The tables below say which run produced them.

In [ ]:
for name, table in report.error_tables(MODEL).items():
    print(f"-- {name} --")
    print(table.to_string(float_format=lambda v: f"{v:,.4f}"))
    print()

## Which features carry the model

Gain importance, averaged over the 24 lead-specific models.

In [ ]:
display(Image(report.plot_feature_importance()))

## Business conclusions

Derived from the grouped tables rather than asserted — rerunning the backtest rewrites them.

In [ ]:
for i, finding in enumerate(report.business_findings(MODEL), 1):
    print(f"{i}. {finding}\n")

## Caveats worth stating before anyone quotes these numbers

1. **Perfect weather forecast.** The `temp`/`hdd`/`cdd` features use realised temperature at
   the target hour. A production system substitutes a numerical weather prediction and gives
   back part of the improvement. This is the single largest optimism in the results.
2. **SARIMA and Prophet train on a truncated window** (2 and 3 years) for tractability across
   52 refits; LightGBM and the baselines use the full expanding window. The comparison is
   therefore favourable to the tree model on data volume, and the truncation is reported.
3. **The operating day is 05:00–05:00 UTC**, i.e. midnight EST, not local midnight. This keeps
   the horizon at exactly 24 points across DST at the cost of a one-hour offset during EDT.
4. **52 folds is one year.** It is enough to average out day-to-day variance; it is not enough
   to say anything about year-to-year drift in model ranking.